Step 0 — Add my info

In [ ]:
first_name = "Eric"
last_name = "Tamele"
ID = 70274
id_20 = 1 + ID % 30

print("First name:", first_name)
print("Last name:", last_name)
print("ID:", ID)
print("id_20:", id_20)

Step 1 — Install PySpark + JDK and create Spark session

In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tamele_SparkSession") \
    .getOrCreate()

spark


Step 2 — Unzip Data.zip and load the five CSV files

In [ ]:
import zipfile

with zipfile.ZipFile("Data.zip", "r") as z:
    z.extractall("data_task5")

import os
os.listdir("data_task5")

execution = spark.read.option("header", True).option("sep", ";").csv("data_task5/execution.csv")
person    = spark.read.option("header", True).option("sep", ";").csv("data_task5/person.csv")
dept      = spark.read.option("header", True).option("sep", ";").csv("data_task5/dept.csv")
position  = spark.read.option("header", True).option("sep", ";").csv("data_task5/position.csv")
action    = spark.read.option("header", True).option("sep", ";").csv("data_task5/action.csv")



Step 3 — Create DataFrame “execution” and show first 5 rows

In [ ]:
execution.show(5)


Step 4 — Create DataFrame “person” and show first 5 rows

In [ ]:
person.show(5)


Step 5 — Create DataFrame “dept” and show first 5 rows

In [ ]:
dept.show(5)


Step 6 — Create DataFrame “position” and show first 5 rows

In [ ]:
position.show(5)


Step 7 — Create DataFrame “action” and show first 5 rows

In [ ]:
action.show(5)


Step 8 — Joined views for your id_20

In [ ]:
from pyspark.sql.functions import col

joined = (
    execution.alias("e")
    .join(person.alias("p"), col("e.id_person") == col("p.id"))
    .join(dept.alias("d"), col("e.id_dept") == col("d.id"))
    .join(position.alias("pos"), col("e.id_position") == col("pos.id"))
    .join(action.alias("a"), col("e.id_execution") == col("a.id"))
    .select(
        col("e.id_20"),
        col("e.time"),
        col("e.cost"),
        col("e.payment"),
        col("p.name"),
        col("d.dept"),
        col("pos.position"),
        col("a.action")
    )
)

joined_id20 = joined.filter(col("id_20") == id_20)

joined_id20.show(5)


Step 8 — Joined views for your id_20

In [ ]:
from pyspark.sql.functions import mean, stddev

count_obs = joined_id20.count()

stats = joined_id20.agg(
    mean("time").alias("mean_time"),
    stddev("time").alias("std_time"),
    mean("cost").alias("mean_cost"),
    stddev("cost").alias("std_cost"),
    mean("payment").alias("mean_payment"),
    stddev("payment").alias("std_payment")
)

print("Number of observations:", count_obs)
stats.show()
